# Assignment 10: Modeling Blood Pressure and Testing Predictions Honestly

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python check_assignment.py` in the terminal.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import statsmodels
import statsmodels.formula.api as smf
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, mean_absolute_error, mean_squared_error,
                             precision_score, r2_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("pandas:", pd.__version__)
print("statsmodels:", statsmodels.__version__)
print("scikit-learn:", sklearn.__version__)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

COEFFICIENTS_PATH = OUTPUT_DIR / "ols_coefficients.csv"
INTERVALS_PATH = OUTPUT_DIR / "new_patient_intervals.csv"
RESIDUALS_PATH = OUTPUT_DIR / "ols_residuals.csv"
RESIDUAL_PLOT_PATH = OUTPUT_DIR / "residuals_vs_fitted.png"
AVAILABILITY_PATH = OUTPUT_DIR / "availability_decisions.csv"
SPLIT_PATH = OUTPUT_DIR / "split_summary.csv"
VALIDATION_PATH = OUTPUT_DIR / "validation_metrics.csv"
TEST_METRICS_PATH = OUTPUT_DIR / "test_metrics.csv"
TEST_PREDICTIONS_PATH = OUTPUT_DIR / "test_predictions.csv"
READMISSION_PATH = OUTPUT_DIR / "readmission_metrics.csv"

print("data folder found:", DATA_DIR.exists())

## Load the data

This cell reads the four supplied files and parses the two visit timestamps as UTC (Lecture 09). Run it as it is.

In [ ]:
patients = pd.read_csv(DATA_DIR / "clinic_bp.csv")
candidates = pd.read_csv(DATA_DIR / "feature_availability.csv")
visits = pd.read_csv(DATA_DIR / "followup_visits.csv")
visits["visit_time"] = pd.to_datetime(visits["visit_time"], utc=True)
visits["followup_time"] = pd.to_datetime(visits["followup_time"], utc=True)
flags = pd.read_csv(DATA_DIR / "readmission_flags.csv")

print("patients:", patients.shape)
print("candidates:", candidates.shape)
print("visits:", visits.shape)
print("flags:", flags.shape)

## Task 1: Model blood pressure with statsmodels

### 1.1 Fit the model and save its coefficients

> **Checkpoint: `output/ols_coefficients.csv`**
> Three rows, `Intercept`, `age`, and `bmi`, under the header `term,coef,std_err,ci_lower,ci_upper`.

In [ ]:
results = None  # TODO: smf.ols("sbp ~ age + bmi", data=patients).fit()
print("rows used:", results.nobs)
print("R-squared:", round(results.rsquared, 3))

coefficients = None  # TODO: pd.DataFrame({...}) with the columns coef (results.params), std_err (results.bse),
#   ci_lower (results.conf_int()[0]), and ci_upper (results.conf_int()[1]); its index holds the three terms
# TODO: name that index with coefficients.index.name = "term"
# TODO: save coefficients to COEFFICIENTS_PATH, keeping the index (leave out index=False)

coefficients

### 1.2 Give a new patient both intervals

> **Checkpoint: `output/new_patient_intervals.csv`**
> One row under the header `age,bmi,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper`.

In [ ]:
new_patient = pd.DataFrame({"age": [60], "bmi": [31.0]})

intervals = None  # TODO: results.get_prediction(new_patient).summary_frame(alpha=0.05)
new_patient_intervals = None  # TODO: pd.concat([new_patient, intervals], axis=1), so age and bmi sit beside the intervals
print("rows:", len(new_patient_intervals))

# TODO: save new_patient_intervals to INTERVALS_PATH without the row index

new_patient_intervals

### 1.3 Check the residuals

> **Checkpoint: `output/ols_residuals.csv`**
> Twenty rows, `P01` to `P20`, under the header `patient_id,observed,fitted,residual`.

> **Checkpoint: `output/residuals_vs_fitted.png`**
> A scatter of residuals against fitted values with a dashed line at 0.

In [ ]:
residual_table = None  # TODO: pd.DataFrame({...}) with the columns patient_id (patients["patient_id"]),
#   observed (patients["sbp"]), fitted (results.fittedvalues), and residual (results.resid)
print("rows:", len(residual_table))

# TODO: save residual_table to RESIDUALS_PATH without the row index

residual_table.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: ax.scatter() with results.fittedvalues on x and results.resid on y
# TODO: a dashed reference line at 0: ax.axhline(0, color="gray", linestyle="--")
# TODO: label the axes "Fitted SBP (mmHg)" and "Residual (observed - fitted)"
# TODO: save with fig.savefig(RESIDUAL_PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

## Task 2: Frame the prediction problem

### 2.1 Audit the candidate features

> **Checkpoint: `output/availability_decisions.csv`**
> Five rows, one per candidate, under the header `candidate_feature,hours_after_visit,available,decision`.

In [ ]:
decisions = None  # TODO: a .copy() of candidates
# TODO: add the column available: decisions["hours_after_visit"] <= 0
# TODO: add the column decision: np.where(decisions["available"], "Keep", "Exclude (leakage)")

# TODO: save decisions to AVAILABILITY_PATH without the row index

FEATURES = None  # TODO: decisions.loc[decisions["available"], "candidate_feature"].tolist()
TARGET = "sbp_followup"
print("features:", FEATURES)

decisions

### 2.2 Split on the target time

> **Checkpoint: `output/split_summary.csv`**
> Three rows, `train`, `validation`, and `test`, under the header `partition,row_count,first_target_time,last_target_time`.

In [ ]:
VALIDATION_START = pd.Timestamp("2026-05-01", tz="UTC")
TEST_START = pd.Timestamp("2026-05-09", tz="UTC")

train = None  # TODO: the visits whose followup_time is before VALIDATION_START
valid = None  # TODO: the visits whose followup_time is on or after VALIDATION_START and before TEST_START
test = None  # TODO: the visits whose followup_time is on or after TEST_START
print("rows:", len(train), len(valid), len(test))

split_summary = None  # TODO: pd.DataFrame({...}) with one row per partition and the columns
#   partition (["train", "validation", "test"]), row_count (each part's len()),
#   first_target_time (each part's followup_time.min()), and last_target_time (each part's followup_time.max())

# TODO: save split_summary to SPLIT_PATH without the row index

split_summary

## Task 3: Compare, freeze, and test once

### 3.1 Compare a baseline and a pipeline on the validation rows

> **Checkpoint: `output/validation_metrics.csv`**
> Two rows, `mean_baseline` and `linear_pipeline`, under the header `approach,mae,rmse,r2`.

In [ ]:
baseline = None  # TODO: DummyRegressor(strategy="mean"), then .fit(train[FEATURES], train[TARGET])
pipeline = None  # TODO: Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]), fitted the same way

validation_rows = []
for name, fitted in [("mean_baseline", baseline), ("linear_pipeline", pipeline)]:
    predicted = None  # TODO: fitted.predict(valid[FEATURES])
    validation_rows.append({
        "approach": name,
        "mae": None,  # TODO: mean_absolute_error(valid[TARGET], predicted)
        "rmse": None,  # TODO: np.sqrt(mean_squared_error(valid[TARGET], predicted))
        "r2": None,  # TODO: r2_score(valid[TARGET], predicted)
    })
validation_metrics = pd.DataFrame(validation_rows)

# TODO: save validation_metrics to VALIDATION_PATH without the row index

winner = None  # TODO: validation_metrics.loc[validation_metrics["mae"].idxmin(), "approach"]
print("frozen choice:", winner)

validation_metrics

### 3.2 Freeze the winner and test it once

> **Checkpoint: `output/test_metrics.csv`**
> One row, `linear_pipeline`, under the header `approach,mae,rmse,r2`.

> **Checkpoint: `output/test_predictions.csv`**
> Ten rows, `V39` to `V48`, under the header `visit_id,followup_time,sbp_followup,predicted_sbp`.

In [ ]:
final = None  # TODO: Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]), the frozen steps
# TODO: fit final on train[FEATURES] and train[TARGET], or refit it on the training plus validation rows:
#   train_valid = pd.concat([train, valid]), then final.fit(train_valid[FEATURES], train_valid[TARGET])
test_predicted = None  # TODO: final.predict(test[FEATURES]), the one test prediction

test_metrics = None  # TODO: pd.DataFrame({...}) with one row: approach "linear_pipeline" and its
#   mae, rmse, and r2 for test[TARGET] and test_predicted, computed as in Task 3.1;
#   put each value in a one-item list, as in "approach": ["linear_pipeline"]
test_predictions = None  # TODO: test[["visit_id", "followup_time", TARGET]].copy()
# TODO: add the column predicted_sbp: test_predicted

# TODO: save test_metrics to TEST_METRICS_PATH and test_predictions to TEST_PREDICTIONS_PATH, both without the row index

test_metrics

### 3.3 Score the supplied readmission flags

> **Checkpoint: `output/readmission_metrics.csv`**
> Two rows, `model_flag` and `never_flag`, under the header `approach,accuracy,precision,recall`.

In [ ]:
readmission_rows = []
for column in ["model_flag", "never_flag"]:
    readmission_rows.append({
        "approach": column,
        "accuracy": None,  # TODO: accuracy_score(flags["readmitted_30d"], flags[column])
        "precision": None,  # TODO: precision_score(flags["readmitted_30d"], flags[column], zero_division=0)
        "recall": None,  # TODO: recall_score(flags["readmitted_30d"], flags[column])
    })
readmission_metrics = pd.DataFrame(readmission_rows)

# TODO: save readmission_metrics to READMISSION_PATH without the row index

readmission_metrics

## Fresh-run check

Click **Restart**, then **Run All**. This cell checks the notebook's own results; `python check_assignment.py` checks the saved files.

In [ ]:
assert results.nobs == 20, "Task 1.1: fit the model on all 20 patients"
assert len(new_patient_intervals) == 1, "Task 1.2: one row for the new patient"
assert len(residual_table) == 20, "Task 1.3: one residual per patient"
assert FEATURES == ["age", "bmi", "sbp_today"], "Task 2.1: keep only the features known when the visit ends"
assert (len(train), len(valid), len(test)) == (30, 8, 10), "Task 2.2: split on followup_time, the target time"
assert winner == "linear_pipeline", "Task 3.1: the lower validation MAE wins"
assert len(test_predictions) == 10, "Task 3.2: one prediction per test visit"
for path in [COEFFICIENTS_PATH, INTERVALS_PATH, RESIDUALS_PATH, RESIDUAL_PLOT_PATH, AVAILABILITY_PATH,
             SPLIT_PATH, VALIDATION_PATH, TEST_METRICS_PATH, TEST_PREDICTIONS_PATH, READMISSION_PATH]:
    assert path.exists(), f"{path} is missing; run the cell that saves it"

print("Fresh-run check passed")